In [15]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import os, io, pickle
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    SpacyTextSplitter
)
import tiktoken
import statistics
import io
import os
import fitz  # PyMuPDF
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI
from google.oauth2.credentials import Credentials


In [16]:
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

def authenticate_google():
    creds = None
    if os.path.exists("token"):
        with open("token", "rb") as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials3.json", SCOPES)
            creds = flow.run_local_server(port=0)
        with open("token", "wb") as token:
            pickle.dump(creds, token)
    
    return build("drive", "v3", credentials=creds)

def download_pdf(file_id, output_path):
    service = authenticate_google()
    request = service.files().get_media(fileId=file_id)
    fh = io.FileIO(output_path, "wb")
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while done is False:
        status, done = downloader.next_chunk()
        print(f"Téléchargement : {int(status.progress() * 100)}%")
    print(f"Fichier téléchargé : {output_path}")


file_id = "1AHE1lXi_kyrtRw31qEGJ7OYE9EO8kfGe"
download_pdf(file_id, "Histoire_CM1.pdf")


Téléchargement : 100%
Fichier téléchargé : Histoire_CM1.pdf


In [ ]:

# Fonction pour estimer le nombre de tokens d’un texte
def count_tokens(text, model="gpt-3.5-turbo"):
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

# Charger ton PDF
loader = PyPDFLoader("Histoire_CM1.pdf")
docs = loader.load()

# Concaténer tout le contenu du PDF en un seul texte brut
full_text = "\n".join([page.page_content for page in docs])

# Définir les splitters à tester
splitters = {
    "RecursiveCharacterTextSplitter": RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50),
    "CharacterTextSplitter": CharacterTextSplitter(separator="\n\n", chunk_size=500, chunk_overlap=50),
    "SpacyTextSplitter": SpacyTextSplitter(pipeline="fr_core_news_sm", chunk_size=500)
}

# Tester les splitters
for name, splitter in splitters.items():
    chunks = splitter.split_text(full_text)
    token_counts = [count_tokens(chunk) for chunk in chunks]

    print(f"\n🧩 {name}")
    print(f" - Nombre de chunks : {len(chunks)}")
    print(f" - Tokens/chunk (moy) : {round(statistics.mean(token_counts))}")
    print(f" - Tokens max : {max(token_counts)}")
    print(f" - Aperçu du 1er chunk :\n{chunks[0][:300]}...")




🧩 RecursiveCharacterTextSplitter
 - Nombre de chunks : 27
 - Tokens/chunk (moy) : 144
 - Tokens max : 161
 - Aperçu du 1er chunk :
LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  
 
 
1)  Qu’est-ce que l’Histoire ? 
 
*L’Histoire est l’étude de notre passé  pour mieux 
comprendre notre vie aujourd’hui. 
 
 *Pour découvrir notre passé, les historiens font des fouilles 
archéologiques, étudient des objets, des documents, des récits...

🧩 CharacterTextSplitter
 - Nombre de chunks : 1
 - Tokens/chunk (moy) : 3826
 - Tokens max : 3826
 - Aperçu du 1er chunk :
LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  
 
 
1)  Qu’est-ce que l’Histoire ? 
 
*L’Histoire est l’étude de notre passé  pour mieux 
comprendre notre vie aujourd’hui. 
 
 *Pour découvrir notre passé, les historiens font des fouilles 
archéologiques, étudient des objets, des documents, des récits...

🧩 SpacyTextSplitter
 - Nombre de chunks : 40
 - Tokens/chunk (moy) : 129
 - Tokens max : 160
 - Aperçu du 1er chunk :
LLeeççoonnss  dd’

In [18]:
encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
tokens = encoding.encode("Ceci est un exemple de phrase.")
print(len(tokens))  # nombre de tokens
print(len(chunks)) #nb chunks sur le dernier

8
40


CharacterTextSplitter = 1 chunk ??

In [19]:
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from transformers import GPT2TokenizerFast
import statistics

# Tokenizer OpenAI-like (GPT-2 compatible)
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

# Fonction pour compter les tokens
def count_tokens(text):
    return len(tokenizer.encode(text))

# Liste des splitters à tester
splitters = {
    "CharacterTextSplitter": CharacterTextSplitter(chunk_size=500, chunk_overlap=50),
    "RecursiveCharacterTextSplitter": RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50),
    "SpacyTextSplitter": SpacyTextSplitter(pipeline="fr_core_news_sm", chunk_size=500),
}

# Texte complet à découper (remplace ceci par ton vrai texte)
full_text = "\n".join([page.page_content for page in docs])

# Analyse automatique
results = []

for name, splitter in splitters.items():
    chunks = splitter.split_text(full_text)
    token_counts = [count_tokens(chunk) for chunk in chunks]

    res = {
        "splitter": name,
        "n_chunks": len(chunks),
        "avg_tokens": round(sum(token_counts)/len(token_counts), 2),
        "min_tokens": min(token_counts),
        "max_tokens": max(token_counts),
        "std_tokens": round(statistics.stdev(token_counts), 2) if len(token_counts)>1 else 0,
        "first_chunk_preview": chunks[0][:300] + "..."
    }

    results.append(res)

# Affichage
for res in results:
    print(f"\n🧩 {res['splitter']}")
    print(f"  - Chunks       : {res['n_chunks']}")
    print(f"  - Moyenne      : {res['avg_tokens']} tokens")
    print(f"  - Min / Max    : {res['min_tokens']} / {res['max_tokens']} tokens")
    print(f"  - Écart-type   : {res['std_tokens']} tokens")
    print(chunks[0])
    print(chunks[1])


Token indices sequence length is longer than the specified maximum sequence length for this model (4989 > 1024). Running this sequence through the model will result in indexing errors



🧩 CharacterTextSplitter
  - Chunks       : 1
  - Moyenne      : 4989.0 tokens
  - Min / Max    : 4989 / 4989 tokens
  - Écart-type   : 0 tokens
LLeeççoonnss  dd’’hhiissttooiirree    CCMM11  
 
 
1)  

Qu’est-ce que l’Histoire ? 
 
*L’

Histoire

est l’étude de notre passé  pour mieux 
comprendre notre vie aujourd’hui.


 
 *Pour découvrir notre passé, les historiens font des fouilles 
archéologiques, étudient des objets, des documents, des récits…  
*Ils représentent le temps par une ligne graduée  : c’est la frise 
chronologique.  
 
*Avant l’invention de l’écriture, c’est la Préhistoire , ensuite vient 
l’Histoire.  


*L’
*L’

Histoire de France est divisée en 5 périodes : 
l’Antiquité

–

le Moyen Âge – les Temps Modernes

– le XIX ème 
siècle

– le XXème siècle. 
 
 
 
 
2)

Des traces du passé : les grottes ornées 
 
*En 1940, 4 enfants découvrent une grotte recouverte 
de peintures  : des taureaux, des cerfs, des chevaux…  : 
la grotte de Lascaux.  

En datant les objets trouvé

In [5]:
%pip install PyPDF2 pandas tqdm openai -q

Note: you may need to restart the kernel to use updated packages.


In [21]:


# Google Drive API scope
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

# 1. Authentification Google Drive
def authenticate_gdrive():
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        with open("token.json", "w") as token:
            token.write(creds.to_json())
    return creds

# 2. Télécharger PDF depuis Drive
def download_pdf_from_drive(service, file_id, destination):
    request = service.files().get_media(fileId=file_id)
    fh = io.FileIO(destination, "wb")
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        print(f"Téléchargement {int(status.progress() * 100)}%")
    fh.close()

# 3. Extraire texte du PDF
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    return full_text

# 4. Chunker le texte
def chunk_text(text, chunk_size=500, chunk_overlap=50):
    splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_text(text)

# 5. Construire l’index vectoriel FAISS
def build_faiss_index(chunks, embeddings_model):
    print(f"Création embeddings pour {len(chunks)} chunks...")
    embeddings = embeddings_model.embed_documents(chunks)
    index = FAISS.from_embeddings(embeddings, chunks)
    return index

# 6. Requête utilisateur + récupération réponse GPT (RAG)
def query_knowledge_base(index, question, llm_model):
    retriever = index.as_retriever(search_kwargs={"k":3})
    qa_chain = RetrievalQA.from_chain_type(llm=llm_model, retriever=retriever)
    answer = qa_chain.run(question)
    return answer

# === MAIN ===
if __name__ == "__main__":
    # Variables à personnaliser
    FILE_ID = "1AHE1lXi_kyrtRw31qEGJ7OYE9EO8kfGe"  # Remplace par l’ID du PDF dans Drive
    LOCAL_PDF = "Histoire_CM1.pdf"
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # Tu dois avoir ta clé dans .env

    # Vérif clé OpenAI
    if not OPENAI_API_KEY:
        raise ValueError("Mets ta clé OPENAI_API_KEY dans le .env et charge-la avant")

    # 1. Authentification Google Drive
    creds = authenticate_gdrive()
    drive_service = build("drive", "v3", credentials=creds)

    # 2. Télécharger PDF
    download_pdf_from_drive(drive_service, FILE_ID, LOCAL_PDF)

    # 3. Extraire texte
    text = extract_text_from_pdf(LOCAL_PDF)
    print(f"Texte extrait ({len(text)} caractères)")

    # 4. Chunking
    chunks = chunk_text(text)
    print(f"Nombre de chunks: {len(chunks)}")

    # 5. Embeddings + index FAISS
    embeddings_model = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
    index = build_faiss_index(chunks, embeddings_model)

    # 6. Initialiser LLM OpenAI
    llm = OpenAI(openai_api_key=OPENAI_API_KEY, model_name="gpt-4")

    # 7. Pose une question de test
    question = "Peux-tu me résumer le contenu ?"
    answer = query_knowledge_base(index, question, llm)

    print(f"\nRéponse du modèle:\n{answer}")


ValueError: Mets ta clé OPENAI_API_KEY dans le .env et charge-la avant

In [23]:
import fitz  # PyMuPDF
import json
from langchain.text_splitter import CharacterTextSplitter

# 1. Extraction du texte du PDF
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    return full_text

# 2. Découpage en chunks
def chunk_text(text, chunk_size=500, chunk_overlap=50):
    splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    chunks = splitter.split_text(text)
    return chunks

# 3. Sauvegarde des chunks dans un JSON
def save_chunks_to_json(chunks, json_path):
    data = []
    for i, chunk in enumerate(chunks):
        data.append({
            "chunk_id": i,
            "text": chunk,
            "level": 0  # tu peux adapter selon ta logique de niveau
        })
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"✅ Sauvegarde de {len(chunks)} chunks dans {json_path}")

# 4. Exemple complet
if __name__ == "__main__":
    pdf_path = "1AHE1lXi_kyrtRw31qEGJ7OYE9EO8kfGe"  # remplace par ton chemin
    json_path = "chunks.json"

    texte = extract_text_from_pdf(pdf_path)
    chunks = chunk_text(texte)
    save_chunks_to_json(chunks, json_path)

    # Affiche un aperçu des premiers chunks
    print("\n--- Aperçu des 3 premiers chunks ---")
    for i, c in enumerate(chunks[:3]):
        print(f"Chunk {i} ({len(c)} caractères):\n{c}\n{'-'*40}")



FileNotFoundError: no such file: '1AHE1lXi_kyrtRw31qEGJ7OYE9EO8kfGe'

In [ ]:
import json
import os
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain.chains import RetrievalQA
from langchain.llms import OpenAI

# Charge ta clé OpenAI dans variable d'environnement OPENAI_API_KEY
# export OPENAI_API_KEY="ta_clef_api"

# 1. Charger les chunks depuis le JSON
def load_chunks_from_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    docs = [Document(page_content=chunk["text"], metadata={"chunk_id": chunk["chunk_id"]}) for chunk in data]
    return docs

# 2. Créer l'index vectoriel (FAISS)
def create_faiss_index(docs, embedding_model):
    embeddings = embedding_model.embed_documents([doc.page_content for doc in docs])
    index = FAISS.from_embeddings(embeddings, docs)
    return index

# 3. Question / réponse avec l’index + LLM OpenAI
def ask_question(index, query):
    # Setup chain retrieval + LLM OpenAI
    llm = OpenAI(temperature=0)
    qa = RetrievalQA.from_chain_type(llm=llm, retriever=index.as_retriever(), return_source_documents=True)
    result = qa.run(query)
    return result

if __name__ == "__main__":
    json_path = "chunks.json"  # Le fichier JSON produit avant
    query = "Donne-moi une question sur le thème de l'école"  # Ta question test

    # Instancie l’embedder OpenAI
    embed_model = OpenAIEmbeddings()

    # Load chunks
    documents = load_chunks_from_json(json_path)

    # Create FAISS index
    index = create_faiss_index(documents, embed_model)

    # Poser la question
    answer = ask_question(index, query)

    print("\n=== Réponse ===")
    print(answer)

